In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

DATA_PATH = Path("../data/raw/load_data.csv")

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head())

In [ ]:
print("Column names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nDataset information:")
df.info()

print("\nNumeric summary:")
display(df.describe().T)

In [ ]:
missing = pd.DataFrame({
    "Missing Count": df.isna().sum(),
    "Missing Percentage": df.isna().mean() * 100
})

display(missing.sort_values(
    "Missing Count", ascending=False
))

plt.figure(figsize=(10, 5))
sns.barplot(
    x=missing.index,
    y=missing["Missing Percentage"]
)
plt.xticks(rotation=75, ha="right")
plt.title("Missing Values by Feature")
plt.ylabel("Missing Percentage (%)")
plt.tight_layout()
plt.show()

In [ ]:
print("Fully duplicated rows:", df.duplicated().sum())

print(
    "Duplicate timestamp strings:",
    df["Date_Time"].duplicated().sum()
)

duplicate_dates = df[
    df["Date_Time"].duplicated(keep=False)
]

display(duplicate_dates)

In [ ]:
df["Date_Time"] = pd.to_datetime(
    df["Date_Time"],
    format="%d-%m-%Y %H:%M",
    errors="coerce"
)

print("Invalid timestamps:", df["Date_Time"].isna().sum())

print("Earliest:", df["Date_Time"].min())
print("Latest:", df["Date_Time"].max())

print(
    "Already chronologically ordered:",
    df["Date_Time"].is_monotonic_increasing
)

monthly_counts = (
    df.groupby(df["Date_Time"].dt.to_period("M"))
      .size()
)

display(monthly_counts)

In [ ]:
print(df["Load_Type"].value_counts(dropna=False))

print("\nClass percentages:")
print(
    df["Load_Type"]
    .value_counts(normalize=True, dropna=False)
    .mul(100)
    .round(2)
)

plt.figure(figsize=(8, 5))
sns.countplot(
    data=df,
    x="Load_Type",
    order=["Light_Load", "Medium_Load", "Maximum_Load"]
)
plt.title("Load Type Distribution")
plt.tight_layout()
plt.show()

In [ ]:
expected_nsm = (
    df["Date_Time"].dt.hour * 3600
    + df["Date_Time"].dt.minute * 60
    + df["Date_Time"].dt.second
)

invalid_nsm_range = (
    (df["NSM"] < 0)
    | (df["NSM"] >= 86400)
)

nsm_mismatch = (
    df["NSM"].notna()
    & expected_nsm.notna()
    & ~np.isclose(df["NSM"], expected_nsm)
)

print("NSM outside daily range:", invalid_nsm_range.sum())
print("NSM inconsistent with timestamp:", nsm_mismatch.sum())

display(
    df.loc[nsm_mismatch, ["Date_Time", "NSM"]].head(20)
)

In [ ]:
test_start = pd.Timestamp("2018-12-01")

train_dev = df[df["Date_Time"] < test_start].copy()

test = df[
    (df["Date_Time"] >= test_start)
    & (df["Date_Time"] < pd.Timestamp("2019-01-01"))
].copy()

print("Development records:", len(train_dev))
print("Final test records:", len(test))

print("\nDevelopment date range:")
print(train_dev["Date_Time"].min(), train_dev["Date_Time"].max())

print("\nTest date range:")
print(test["Date_Time"].min(), test["Date_Time"].max())